<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# Navigating the Filesystem

This tutorial introduces how ROS 2 code is organised on disk and the command-line tools that find it: `ros2 pkg`, `ros2 interface` and `colcon list`.

## Quick Overview of Filesystem Concepts

- **Packages:** Packages are the software organization unit of ROS 2 code. Each package can contain libraries, executables, scripts, launch files, configuration files or other artifacts.
- **Manifests (`package.xml` files):** A manifest is a description of a package. It defines dependencies between packages and meta information about the package itself, such as version, maintainer, license, etc.
- **Interfaces:** message (`.msg`), service (`.srv`) and action (`.action`) definitions. They also live in packages (for example `std_msgs`, `sensor_msgs` and the Duckietown `duckietown_msgs`).
- **Workspaces:** a directory in which packages are built and installed together. An installed package is registered in the **ament index**, which is what the `ros2` tools search.

## Follow the tutorial

Use the menu on the top-left corner of this editor, then `Terminal  ->  New Terminal` to open a terminal at the bottom of the page. Use the terminal to execute the commands introduced in this notebook. Feel free to experiment with this terminal as much as you need to get comfortable with these new commands.

Every terminal of this editor already has ROS 2 loaded (the editor runs `source /opt/ros/jazzy/setup.bash` for you, together with the Duckietown interfaces). You can check it with:

```bash
printenv ROS2_DISTRO
```

which prints `jazzy`, the ROS 2 distribution used in this learning experience.

## Filesystem Tools

Source code is usually spread across many ROS 2 packages even for small projects. Navigating with traditional command-line tools such as `ls` and `cd` can be very tedious, because installed packages live in several places (`/opt/ros/jazzy`, the Duckietown interfaces, your own workspace). This is why ROS 2 provides tools to help you navigate the filesystem.

### Using `ros2 pkg`

The command `ros2 pkg` gives you information about packages. In this tutorial we cover four of its sub-commands.

#### `ros2 pkg list`

```bash
ros2 pkg list
```

prints the names of all the packages that ROS 2 can currently see (about two hundred). To find a specific one, filter the list with `grep`:

```bash
ros2 pkg list | grep -E "rclpy|duckietown"
```

You should see at least:

    duckietown_msgs
    rclpy

`rclpy` is the Python client library of ROS 2 (you will import it in your nodes) and `duckietown_msgs` contains the message definitions of the Duckietown stack.

#### `ros2 pkg prefix`

The `prefix` sub-command returns the directory in which a package is installed:

```bash
ros2 pkg prefix rclpy
```

would return:

    /opt/ros/jazzy

while

```bash
ros2 pkg prefix duckietown_msgs
```

returns:

    /opt/duckietown/ros2/duckietown_msgs

The Duckietown messages are installed in a separate **overlay** on top of the ROS 2 distribution. The files of a package are in `<prefix>/share/<package>`; for example, we can list the message definitions of `duckietown_msgs` with:

```bash
ls "$(ros2 pkg prefix duckietown_msgs)/share/duckietown_msgs/msg"
```

#### `ros2 pkg xml`

The `xml` sub-command prints the manifest (`package.xml`) of a package:

```bash
ros2 pkg xml duckietown_msgs
```

Look for the `<name>`, `<description>` and `<depend>` entries: this is how a package declares what it needs.

#### `ros2 pkg executables`

Finally, `executables` lists the programs a package installs (the ones you can start with `ros2 run`, which we will meet in a later notebook):

```bash
ros2 pkg executables rmw_zenoh_cpp
```

returns:

    rmw_zenoh_cpp rmw_zenohd

the zenoh router of the ROS 2 middleware used in Duckietown (notebook 04 is about it). Most packages of the distribution are libraries and install no executable at all; the packages of this learning experience install three (notebook 06).

### Where does ROS 2 look? The `AMENT_PREFIX_PATH`

The `ros2 pkg` tools only find packages that are registered in the **ament index** of a prefix listed in your `AMENT_PREFIX_PATH` environment variable. To see what is in your `AMENT_PREFIX_PATH`, execute the command:

```bash
echo ${AMENT_PREFIX_PATH}
```

You should see a colon-separated list of directories, for example:

    /opt/duckietown/ros2/duckietown_msgs:/opt/ros/jazzy

Every time you `source` the `setup.bash` of a workspace, that workspace is added in front of the list. This is how your own packages become visible to ROS 2 once they are built (notebook 06).

The index itself is just a directory of marker files, one per package:

```bash
ls /opt/ros/jazzy/share/ament_index/resource_index/packages | head
```

## Using `ros2 interface`

Interfaces (messages, services, actions) have their own tool. `ros2 interface list` prints every interface that is installed; `ros2 interface show` prints the definition of one of them:

```bash
ros2 interface list | grep duckietown_msgs/msg | head
ros2 interface show duckietown_msgs/msg/WheelsCmdStamped
```

The second command returns:

    std_msgs/Header header
    	builtin_interfaces/Time stamp
    		int32 sec
    		uint32 nanosec
    	string frame_id
    float32 vel_left
    float32 vel_right

This is the message that commands the two wheels of the Duckiebot. We will come back to interfaces in notebook 03.

## Packages that are not installed yet: `colcon list`

The packages of this learning experience live in the `packages/` directory of this repository (`/code/src/lx-ros2-basics/packages` inside the editor). They are source code: nothing is installed yet, so `ros2 pkg` does not know them. The build tool `colcon` can still discover them. From the root of this repository, run:

```bash
cd /code/src/lx-ros2-basics
colcon list --base-paths packages
```

You should see:

    ros2_basics	packages/ros2_basics	(ros.ament_python)

The three entries identify the package name, its source directory and its package type (`ament_python`: a pure Python ROS 2 package). Notebook 05 looks inside it.

## Review

You may have noticed a pattern with the naming of the ROS 2 commands:

- `ros2 pkg` = **ros2** + `pack`(age)
- `ros2 interface` = **ros2** + interface (messages, services, actions)
- `ros2 node`, `ros2 topic`, `ros2 param`, `ros2 service`, `ros2 run`, `ros2 launch`: every ROS 2 tool is a verb of the single `ros2` command. Type `ros2 --help` to see them all, and `ros2 <verb> --help` for the sub-commands.

Everything these tools find is registered in the ament index of a prefix listed in `AMENT_PREFIX_PATH`.

Let's move to [the next notebook about the ROS 2 graph](../notebooks/02_ros2_graph_and_software_layers.ipynb).